# Validation - checking nanopubs against the conformance shapes

Validates a nanopublication against [`shapes/conformance.shacl.ttl`](../shapes/conformance.shacl.ttl)
with [pySHACL](https://github.com/RDFLib/pySHACL). Validation is a pure RDF + SHACL operation, so this
needs only `rdflib` and `pyshacl` - no signing, profile, or nanopub library.


In [1]:
import sys, subprocess
try:
    import rdflib, pyshacl
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdflib", "pyshacl"])
import pyshacl
from rdflib import Dataset


## A nanopublication to validate

A nanopublication is four named graphs (Head, assertion, provenance, pubinfo). `nanopub(assertion)`
wraps a given assertion in that structure with a minimal, valid provenance and pubinfo; `validate` runs
the shapes over the whole thing and returns whether it conforms plus any messages.


In [2]:
SHAPES = "../shapes/conformance.shacl.ttl"

PREFIXES = """@prefix this:   <https://example.org/np/ex> .
@prefix sub:    <https://example.org/np/ex/> .
@prefix np:     <http://www.nanopub.org/nschema#> .
@prefix schema: <https://schema.org/> .
@prefix rdf:    <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix cito:   <http://purl.org/spar/cito/> .
@prefix prov:   <http://www.w3.org/ns/prov#> .
@prefix dct:    <http://purl.org/dc/terms/> .
@prefix xsd:    <http://www.w3.org/2001/XMLSchema#> .
@prefix orcid:  <https://orcid.org/> ."""

def nanopub(assertion):
    return f"""{PREFIXES}
sub:Head {{
  this: a np:Nanopublication ;
    np:hasAssertion sub:assertion ; np:hasProvenance sub:provenance ;
    np:hasPublicationInfo sub:pubinfo .
}}
sub:assertion {{
{assertion}
}}
sub:provenance {{
  sub:assertion prov:wasAttributedTo orcid:0000-0001-2345-6789 .
}}
sub:pubinfo {{
  this: dct:creator orcid:0000-0001-2345-6789 ;
    dct:license <https://creativecommons.org/licenses/by/4.0/> ;
    dct:created "2026-07-14T00:00:00Z"^^xsd:dateTime .
}}"""

def validate(assertion):
    ds = Dataset(); ds.parse(data=nanopub(assertion), format="trig")
    conforms, _, text = pyshacl.validate(ds, shacl_graph=SHAPES)
    messages = [line.split("Message:", 1)[1].strip()
                for line in text.splitlines() if "Message:" in line]
    return conforms, messages


## A conforming nanopublication

A discourse contribution: a `schema:Statement` with a value and a CiTO stance.


In [3]:
conforms, messages = validate("""  sub:claim a schema:Statement ;
    rdf:value     "Open peer review improves the quality of published critiques." ;
    cito:supports <https://w3id.org/np/RAexample/claim> .""")

print("conforms:", conforms)
print("messages:", messages)


conforms: True
messages: []


## A failing nanopublication

The same claim with `rdf:value` removed. `DiscourseContribution` requires exactly one, so validation
fails and reports why.


In [4]:
conforms, messages = validate("""  sub:claim a schema:Statement ;
    cito:supports <https://w3id.org/np/RAexample/claim> .""")

print("conforms:", conforms)
for m in messages:
    print("  -", m)


conforms: False
  - A discourse contribution needs exactly one rdf:value string.


## Note: validate the whole nanopublication

The `AssertionProvenance` and `Nanopublication` shapes target nodes in the provenance and pubinfo
graphs, so validating a single extracted graph would miss them. Validate all four graphs together, as
`nanopub()` produces here.
